In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, pipeline
import torch
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import torch.nn.functional as F

print("IP 3 started: sincerity score (S) module...")

# -------------------------------------------------------------------------
# STEP 1: PREPARE THE INPUTS
# -------------------------------------------------------------------------
# 1. Example motivation letter (the AIESEC section of your CV)
sample_motivation_letter = """
I would like to do activities and things that makes me more self-confident. I like to travel too. 
I wanted to connect these two things together thats why I decided to become Global Volunteer of AIESEC. 
I was working at school with kids on project Global Classroom in Iasi Romania. 
I lived there for 2 months with people from all over the world.
"""

# 2. Load the projects from IP 2
try:
    df_result = pd.read_pickle("IP2_output_listings_M_scored.pkl")
    print(f"-> {len(df_result)} scored projects loaded.")
except FileNotFoundError:
    print("ERROR: 'IP2_output_listings_M_scored.pkl' not found.")
    raise

# -------------------------------------------------------------------------
# STEP 2: CHEATING DETECTION (AI DETECTION) MODEL
# -------------------------------------------------------------------------
print("\nStep 2: Loading the cheating detection (AI detector) model...")
# This model measures the probability that the text was written by AI (GPT).
ai_model_name = "roberta-base-openai-detector"
ai_tokenizer = AutoTokenizer.from_pretrained(ai_model_name)
ai_model = AutoModelForSequenceClassification.from_pretrained(ai_model_name)

def detect_ai_generated(text):
    """Returns the probability (0-1) that the text was written by AI."""
    inputs = ai_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = ai_model(**inputs).logits
    probs = F.softmax(logits, dim=1)
    # 0: Human, 1: AI (Fake=1 according to the model's label structure)
    ai_probability = probs[0][1].item() 
    return ai_probability

# Compute the AI score of the example letter
ai_score = detect_ai_generated(sample_motivation_letter)
print(f"-> Probability that the letter is AI-generated: %{ai_score * 100:.2f}")
# We take the complement for the sincerity score (human-written = high score)
human_score = (1 - ai_score) * 100

# -------------------------------------------------------------------------
# STEP 3: CONTEXTUAL SERIOUSNESS (CONTEXTUAL SIMILARITY)
# -------------------------------------------------------------------------
print("\nStep 3: Loading the contextual similarity model (XLM-R)...")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
embed_model_name = "xlm-roberta-base"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name).to(device)

def get_embedding(text):
    inputs = embed_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = embed_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Extract the letter's vector
letter_vector = get_embedding(sample_motivation_letter)

def calculate_s_score(row):
    # 1. Get the vector of the project description (or recompute it)
    # We did not save the 'full_description' vector in IP 1, so we compute it quickly here
    project_text = row['full_description']
    project_vector = get_embedding(str(project_text))
    
    # 2. Cosine similarity (letter vs project description)
    similarity = cosine_similarity(letter_vector, project_vector)[0][0]
    
    # 3. Final S score: 50% human-written + 50% project similarity
    # (Similarity is usually between 0 and 1, we multiply by 100)
    context_score = similarity * 100
    
    # Formula: S = (Human_Score * 0.4) + (Context_Score * 0.6)
    # Context is more decisive because every letter may be human-written but irrelevant.
    final_s = (human_score * 0.4) + (context_score * 0.6)
    
    return final_s, human_score, context_score

# -------------------------------------------------------------------------
# STEP 4: COMPUTATION AND RESULTS
# -------------------------------------------------------------------------
print("\nComputing sincerity scores (for the first 20 projects)...")

# Score only the best 20 projects for performance
df_top20 = df_result.head(20).copy()

# The apply function will return several columns
metrics = df_top20.apply(calculate_s_score, axis=1, result_type='expand')
df_top20[['S_Score', 'Human_Score', 'Context_Score']] = metrics

# Show the results
print("\n🏆 MOST SINCERE MATCHES (S-Score) 🏆")
print(f"Reference letter AI score: {ai_score:.4f} (lower is better)")
print("-" * 80)

for i, row in df_top20.head(5).iterrows():
    project_title_text = row['link'].split('/')[-1].replace('_en', '').upper()
    print(f"{i+1}. [S-Score: {row['S_Score']:.2f}] {project_title_text}")
    print(f"   (Human-written: {row['Human_Score']:.1f} | Project fit: {row['Context_Score']:.1f})")
    print(f"   M-Score (previous): {row['M_Score']:.2f}")
    print("-" * 80)

# Save
df_top20.to_pickle("IP3_output_listings_S_scored.pkl")
print("\nFile saved: 'IP3_output_listings_S_scored.pkl'")

IP 3 started: sincerity score (S) module...
-> 952 scored projects loaded.

Step 2: Loading the cheating detection (AI detector) model...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-base-openai-detector were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 4216d318-2032-428f-8b98-61c759189cde)')' thrown while requesting HEAD https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


-> Probability that the letter is AI-generated: %99.95

Step 3: Loading the contextual similarity model (XLM-R)...

Computing sincerity scores (for the first 20 projects)...

🏆 MOST SINCERE MATCHES (S-Score) 🏆
Reference letter AI score: 0.9995 (lower is better)
--------------------------------------------------------------------------------
149. [S-Score: 59.73] 48828
   (Human-written: 0.0 | Project fit: 99.5)
   M-Score (previous): 99.87
--------------------------------------------------------------------------------
348. [S-Score: 59.87] 47877
   (Human-written: 0.0 | Project fit: 99.8)
   M-Score (previous): 99.86
--------------------------------------------------------------------------------
204. [S-Score: 59.80] 48664
   (Human-written: 0.0 | Project fit: 99.6)
   M-Score (previous): 99.86
--------------------------------------------------------------------------------
826. [S-Score: 59.90] 33631
   (Human-written: 0.0 | Project fit: 99.8)
   M-Score (previous): 99.85
----------

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import numpy as np

print("IP 4 started: reliability score (R) module...")

# -------------------------------------------------------------------------
# STEP 1: PREPARE THE DATA (CLAIM vs EVIDENCE)
# -------------------------------------------------------------------------
try:
    # 1. Claims: the candidate's skill list
    df_skills = pd.read_pickle("IP1_output_candidate_extracted_skills.pkl")
    candidate_claims = df_skills["extracted_skill_name"].tolist()
    print(f"-> {len(candidate_claims)} 'claims' (skills) loaded.")

    # 2. Evidence: the full text of the CV (including certificates)
    with open("IP1_output_candidate_full_text.txt", "r", encoding="utf-8") as f:
        # We split the text into paragraphs so that the model is not overwhelmed
        cv_paragraphs = [p for p in f.read().split('\n') if len(p) > 20]
    print(f"-> {len(cv_paragraphs)} 'evidence pieces' (CV lines) loaded.")

except FileNotFoundError:
    print("ERROR: IP 1 outputs not found.")
    raise

# -------------------------------------------------------------------------
# STEP 2: NLI (NATURAL LANGUAGE INFERENCE) MODEL
# -------------------------------------------------------------------------
print("\nStep 2: Loading the NLI (digital detective) model...")
# This model takes an (evidence, claim) pair and gives an 'entailment' (verification) score.
nli_model_name = "cross-encoder/nli-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
model = AutoModelForSequenceClassification.from_pretrained(nli_model_name)

# M2 GPU usage
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
print(f"Model loaded on device {device}.")

def verify_claim(claim, evidence_list):
    """Verifies whether a claim (skill) is supported by a list of evidence."""
    
    # Turn the claim into a sentence: "I have the skill of Python."
    hypothesis = f"I have the skill of {claim}."
    
    # Build pairs with the evidence: [("Certificate: Python 101", "I know Python"), ...]
    pairs = [(evidence, hypothesis) for evidence in evidence_list]
    
    # Feed the model
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # NLI model output: [Contradiction, Entailment, Neutral] (the order may differ by model)
    # Labels for cross-encoder/nli-distilroberta-base:
    # 0: Contradiction (refutes)
    # 1: Entailment (confirms) <-- what we are looking for
    # 2: Neutral (unrelated)
    
    # Take the 'entailment' score (index 1) for each evidence line
    entailment_scores = logits[:, 1].cpu().numpy()
    
    # Find the highest verification score (one confirming line is enough)
    max_score = entailment_scores.max()
    
    # Map to a 0-100 score with a sigmoid-like transform (logit -> probability)
    # Simply put: a positive logit means strong confirmation.
    probability = 1 / (1 + np.exp(-max_score)) 
    
    return probability * 100

# -------------------------------------------------------------------------
# STEP 3: VERIFY ALL SKILLS (RELIABILITY SCORE)
# -------------------------------------------------------------------------
print("\nStep 3: Verifying the claims (this may take a while)...")

verification_results = []
total_score = 0

for skill in candidate_claims:
    # Scan the evidence in the CV for each skill
    score = verify_claim(skill, cv_paragraphs)
    total_score += score
    
    status = "✅ Verified" if score > 50 else "❓ Weak evidence"
    verification_results.append({"Skill": skill, "R_Score": score, "Status": status})
    print(f"   Skill: {skill.ljust(30)} -> Score: {score:.1f} ({status})")

# OVERALL RELIABILITY SCORE (R)
# Mean verification rate of all skills
final_r_score = total_score / len(candidate_claims)

print("-" * 60)
print(f"🏆 FINAL RELIABILITY SCORE (R): {final_r_score:.2f} / 100")
print("-" * 60)

# Save the results (IP 5 needs only a single number, but we also keep the details)
df_r_details = pd.DataFrame(verification_results)
df_r_details.to_pickle("IP4_output_R_details.pkl")

# Save the R score to a file as a single number (or keep it in a variable)
with open("IP4_output_R_score.txt", "w") as f:
    f.write(str(final_r_score))

print("Files saved.")

IP 4 started: reliability score (R) module...
-> 13 'claims' (skills) loaded.
-> 3 'evidence pieces' (CV lines) loaded.

Step 2: Loading the NLI (digital detective) model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Model loaded on device mps.

Step 3: Verifying the claims (this may take a while)...
   Skill: social medium                  -> Score: 93.7 (✅ Verified)
   Skill: c                              -> Score: 89.6 (✅ Verified)
   Skill: html                           -> Score: 62.7 (✅ Verified)
   Skill: sql server                     -> Score: 74.1 (✅ Verified)
   Skill: c++                            -> Score: 76.9 (✅ Verified)
   Skill: microsoft powerpoint           -> Score: 32.4 (❓ Weak evidence)
   Skill: microsoft excel                -> Score: 24.1 (❓ Weak evidence)
   Skill: microsoft office               -> Score: 25.8 (❓ Weak evidence)
   Skill: programming languages          -> Score: 87.5 (✅ Verified)
   Skill: microsoft word                 -> Score: 39.4 (❓ Weak evidence)
   Skill: network engineer               -> Score: 93.9 (✅ Verified)
   Skill: zoom                           -> Score: 45.6 (❓ Weak evidence)
   Skill: google drive                   -> Score: 14.8 (❓ Wea

In [3]:
import pandas as pd
import numpy as np

print("IP 5 started: fair ranking (MOO) module integration...")

# -------------------------------------------------------------------------
# STEP 1: BRING ALL SCORES TOGETHER
# -------------------------------------------------------------------------
try:
    # 1. Load the table with the M and S scores (from IP 3)
    df_final = pd.read_pickle("IP3_output_listings_S_scored.pkl")
    print(f"-> {len(df_final)} projects with M and S scores loaded.")
    
    # 2. Load the R score (single value from IP 4)
    with open("IP4_output_R_score.txt", "r") as f:
        r_score = float(f.read())
    print(f"-> Candidate reliability score (R) loaded: {r_score:.2f}")
    
except FileNotFoundError:
    print("ERROR: Output files of the previous steps not found.")
    raise

# -------------------------------------------------------------------------
# STEP 2: MULTI-OBJECTIVE UTILITY FUNCTION
# -------------------------------------------------------------------------
# Formula: Final_Score = (w1 * M) + (w2 * S) + (w3 * R)
# Weights (by scenario):
# - Matching (M): 50% (most important)
# - Sincerity (S): 30% (intent matters)
# - Reliability (R): 20% (CV accuracy)

w_m = 0.5
w_s = 0.3
w_r = 0.2

print(f"\nWeights: M=%{w_m*100}, S=%{w_s*100}, R=%{w_r*100}")

# Since the R score is a single value, it has the same effect on all projects (a property of the candidate)
# But including it in the formula determines reliable candidates' place in the overall ranking (relative to other candidates).
# Since our prototype has a single candidate, it does not change the ranking but it affects the score.

def calculate_moo_score(row):
    m = row['M_Score']
    s = row['S_Score']
    r = r_score
    
    final_score = (m * w_m) + (s * w_s) + (r * w_r)
    return final_score

df_final['Final_Score'] = df_final.apply(calculate_moo_score, axis=1)

# -------------------------------------------------------------------------
# STEP 3: FAIRNESS ADJUSTMENT (SIMULATION)
# -------------------------------------------------------------------------
# CPFair logic:
# Increase the visibility of unpopular or disadvantaged projects.
# Simulation: give a "visibility bonus" to projects related to "culture" or "art".

def apply_fairness_boost(row):
    bonus = 0
    # Support the project if its description contains 'culture' or 'art' and its score is borderline
    text = row['full_description'].lower()
    if 'culture' in text or 'art' in text:
        bonus = 5.0 # +5 points fairness bonus
        
    return row['Final_Score'] + bonus

df_final['Fair_Score'] = df_final.apply(apply_fairness_boost, axis=1)

# -------------------------------------------------------------------------
# STEP 4: SORT AND REPORT THE RESULTS
# -------------------------------------------------------------------------
df_final = df_final.sort_values(by='Fair_Score', ascending=False)

print("\n🚀 FINAL FAIR RANKING RESULTS (TOP 5) 🚀")
print("-" * 100)
print(f"{'PROJECT NAME':<50} | {'M-SCORE':<8} | {'S-SCORE':<8} | {'R-SCORE':<8} | {'FINAL':<8}")
print("-" * 100)

for i, row in df_final.head(5).iterrows():
    project_name = row['link'].split('/')[-1].replace('_en', '').replace('-', ' ').upper()[:45]
    print(f"{project_name:<50} | {row['M_Score']:.1f}     | {row['S_Score']:.1f}     | {r_score:.1f}     | {row['Fair_Score']:.2f}")

# Save the results
df_final.to_csv("IP5_final_fair_ranking.csv", index=False)
print("\n✅ Project completed successfully! Results saved to 'IP5_final_fair_ranking.csv'.")

IP 5 started: fair ranking (MOO) module integration...
-> 20 projects with M and S scores loaded.
-> Candidate reliability score (R) loaded: 58.49

Weights: M=%50.0, S=%30.0, R=%20.0

🚀 FINAL FAIR RANKING RESULTS (TOP 5) 🚀
----------------------------------------------------------------------------------------------------
PROJECT NAME                                       | M-SCORE  | S-SCORE  | R-SCORE  | FINAL
----------------------------------------------------------------------------------------------------
33631                                              | 99.9     | 59.9     | 58.5     | 84.59
47830                                              | 99.8     | 59.9     | 58.5     | 84.59
47877                                              | 99.9     | 59.9     | 58.5     | 84.59
37127                                              | 99.8     | 59.9     | 58.5     | 84.59
26159                                              | 99.8     | 59.9     | 58.5     | 84.58

✅ Project completed su